# HITL review + coaching chats

**Focus:** approve / revise / reject, History `pkg-…`, and the two coaches.

**Open beside this:**
- `careercrew/review.py`
- `careercrew/session_store.py`
- `careercrew/coach.py`
- `app.py` approve / reject / coach handlers
- `docs/CODE_PATH_HITL.md`, `docs/CODE_PATH_COACH.md`


## Say this — HITL (20 seconds)

> After Analyze, the package enters a human-in-the-loop state machine. You can **approve**, **request revision**, or **reject**. Every package gets an id like `pkg-…` stored in SQLite and listed under History. CareerCrew still never auto-submits applications.


## Review state machine (simplified)

```
          submit_for_review
                 │
                 ▼
            PENDING
           /   |   \
     approve revise reject
         │     │     │
         ▼     ▼     ▼
     APPROVED REVISION REJECTED
                 │
                 └──► user re-analyzes → new package
```

Allowed transitions live in `review.py` (`_ALLOWED`).


In [ ]:
# Illustrative state machine (see review.py for the real _ALLOWED map)

ALLOWED = {
    "PENDING": {"APPROVED", "REVISION_REQUESTED", "REJECTED"},
    "REVISION_REQUESTED": {"PENDING"},  # after re-analyze / resubmit path
    "APPROVED": set(),
    "REJECTED": set(),
}

def transition(state: str, action: str) -> str:
    mapping = {
        "approve": "APPROVED",
        "revise": "REVISION_REQUESTED",
        "reject": "REJECTED",
        "resubmit": "PENDING",
    }
    nxt = mapping[action]
    print(f"{state} --{action}--> {nxt}")
    return nxt

state = "PENDING"
state = transition(state, "revise")
state = transition(state, "resubmit")
state = transition(state, "approve")


## Say this — Coaches (20 seconds)

> Two Gradio chat tabs. The **prep coach** helps improve the resume and preparation using Analyze memory. The **post-interview coach** has the full mock transcript, per-question evaluations, and overall score. Both use the chat LLM with guardrailed replies — still advisory, not auto-apply.


## Coach memory bundles

| Coach | Typical context |
|-------|-----------------|
| Prep coach | Profile, gaps, rewrites, prep plan |
| Post-interview coach | Same + interview turns + evaluations + overall score |

Implementation: `careercrew/coach.py` + Gradio handlers in `app.py`.
Chats persist via `session_store.py`.


In [ ]:
# What "full memory" means for the post-interview coach (conceptual)

post_interview_context = {
    "role_title": "Backend Engineer",
    "gaps": ["system design", "incident ownership"],
    "prep_plan_highlights": ["drill STAR stories for ownership"],
    "transcript": [
        {"role": "interviewer", "text": "Tell me about an API you owned."},
        {"role": "candidate", "text": "I owned checkout API and cut p95 40%."},
    ],
    "evaluations": [{"score": 78, "improvements": ["Add tradeoffs"]}],
    "overall_score": 78,
}

print("Post-interview coach can see:")
for k, v in post_interview_context.items():
    kind = type(v).__name__
    shown = v if not isinstance(v, list) else f"{len(v)} items"
    print(f"  • {k}: {kind} = {shown}")


## Code checklist

| Concern | File | Symbol |
|---------|------|--------|
| State machine | `review.py` | approve / revise / reject helpers |
| History UI | `app.py` | History tab + `_tracker_rows` |
| SQLite | `session_store.py` | packages, chats, interviews |
| Coaches | `coach.py` | prep + post-interview reply builders |

**Back to voice detail:** `01_voice_and_transcript.ipynb`
